# YouTube transcript search in Python with Arcmira

[Arcmira](https://arcmira.com) · [API docs](https://arcmira.com/docs) · [Open in Colab](https://colab.research.google.com/github/arcmira/integrations/blob/master/examples/quote-research-notebook/youtube-transcript-search.ipynb)

Run all cells to turn five saved passages about vibe coding into a CSV you can use to check quotations. Open each original recording at its timestamp and listen before and after the passage. The channel identifies the publisher. Confirm who is speaking before attributing a quote.

The default run downloads the public [quote timeline](https://arcmira.github.io/integrations/quote-timeline/) sample and needs no Arcmira key. To research another phrase, change the settings below and rerun all cells. Live mode makes one search across indexed YouTube and podcast transcripts, with up to five results. [Search reference](https://arcmira.com/docs/search) · [Usage](https://arcmira.com/docs/usage-and-billing)

In [1]:
live_search = False
query = "vibe coding"  # Used only in live mode

## Load the passages

Saved mode reads the five selections published on October 11, 2026. Live mode asks for a key privately or uses the runtime's `ARCMIRA_API_KEY`. Keep keys out of notebook cells and shared outputs.

In [2]:
import json
import os
from getpass import getpass
from http.client import HTTPSConnection
from urllib.parse import urlencode
from urllib.request import urlopen


def saved_passages():
    fixture_url = (
        "https://raw.githubusercontent.com/arcmira/integrations/"
        "c4b8831461adc2c6d5ff6d774527c1820142ba7a/examples/quote-timeline/fixture.js"
    )
    with urlopen(fixture_url, timeout=20) as response:
        fixture = response.read().decode("utf-8")
    _, marker, payload = fixture.partition("window.QUOTE_FIXTURE = ")
    if not marker:
        raise ValueError("The saved quote fixture is missing its data.")
    saved = json.loads(payload.strip().removesuffix(";"))
    rows = [
        {"channel": q["channel"], "video": q["title"], "date": q["publishedAt"],
         "seconds": q["passageStartSeconds"], "passage": q["quote"],
         "source": q["youtubeUrl"], "transcript": q["sourceUrl"],
         "transcript_type": "Unspecified in saved example"}
        for q in saved["quotes"]
    ]
    return saved["query"], rows, saved["selection"] + " " + saved["timing"]


def search_passages(phrase):
    key = os.environ.get("ARCMIRA_API_KEY") or getpass("Arcmira API key: ").strip()
    connection = HTTPSConnection("api.arcmira.com", timeout=30)
    try:
        params = urlencode({"q": phrase, "limit": 5})
        connection.request("GET", "/v1/search?" + params,
                           headers={"Authorization": "Bearer " + key})
        response = connection.getresponse()
        if response.status != 200:
            raise RuntimeError(f"Search returned HTTP {response.status}. See https://arcmira.com/docs/errors")
        result = json.load(response)
    finally:
        connection.close()
    rows = []
    for q in result["chunks"]:
        seconds = q.get("line_start_seconds")
        if seconds is None:
            seconds = q["start_seconds"]
        rows.append(
            {"channel": q.get("channel_name") or "", "video": q.get("video_title") or q["video_id"],
             "date": q.get("published_at") or "", "seconds": seconds, "passage": q["text"],
             "passage_start_seconds": q["start_seconds"], "line_start_seconds": q.get("line_start_seconds"),
             "source": "https://www.youtube.com/watch?" + urlencode({"v": q["video_id"], "t": int(seconds)}),
             "transcript": q["watch_url"],
             "transcript_type": q.get("source_label") or q.get("source") or "Unspecified"}
        )
    return phrase, rows, result.get("note") or ""


search_term, passages, scope = search_passages(query) if live_search else saved_passages()
print(f"{len(passages)} passages for {search_term!r}.", scope)

5 passages for 'vibe coding'. Five editorial selections from 2026 publications, not a ranked or exhaustive sample. Whole-second playback pins aligned near sentence starts using source caption cues. Not word-level alignment.


## Check the recording

Live links start at the matched line when available, otherwise at the excerpt start. Saved selections retain their original timestamps. In live mode, excerpts can include several speakers. Keep a passage only after checking its wording and context against the recording.

In [3]:
from html import escape
from IPython.display import HTML, display

table = []
for item in passages:
    minutes, seconds = divmod(int(item["seconds"]), 60)
    url = escape(item["source"], quote=True)
    table.append(
        f'<tr><td>{escape(item["channel"])}</td><td>{escape(item["video"])}</td>'
        f'<td><a href="{url}">{minutes}:{seconds:02}</a></td>'
        f'<td>{escape(item["passage"])}</td></tr>'
    )
if table:
    display(HTML("<table><tr><th>Channel</th><th>Recording</th><th>Play</th>"
                 "<th>Passage</th></tr>" + "".join(table) + "</table>"))
else:
    print("No passages returned. Try another phrase or check the coverage note above.")

Channel,Recording,Play,Passage
Masters of Scale,60% of today's jobs didn't exist in 1940. 5 leaders on what's next.,13:43,Vibe coding is is essentially using an AI chatbot to bring your ideas to life.
TBPN,"Microsoft, Meta, and Tesla Earnings, Hill & Valley Preview, ChatGPT Ads with Eric Seufert",87:06,"It feels like the the the solution to vibe coding is more vibe coding or more code, right?"
Monitor The Situation,"Greg Eisenberg on Codex, Agentic Engineering, and the Chief Clipping Officer | MTS Live",21:21,"So, um yeah, I think the state is uh vibe coding is almost like the gateway drug to agentic engineering."
Lenny's Podcast,The rise of the professional vibe coder (a new AI-era job),0:00,I'm the first official vibe coding engineer at Lovable.
Theo - t3.gg,You're falling behind. It's time to catch up.,8:21,All these things that were really annoying before it can just do. It's kind of wild.


## Export the worksheet

This writes `arcmira-quote-sources.csv` in the notebook's working directory. Open the **Files** panel in Colab or Jupyter to download it. Use the review column to record corrections or attribution after listening. Formula-like text receives a leading apostrophe in the CSV so a spreadsheet treats it as text.

In [4]:
import csv
from pathlib import Path


def spreadsheet_text(value):
    if isinstance(value, str) and value.lstrip().startswith(("=", "+", "-", "@")):
        return "'" + value
    return value


output = Path("arcmira-quote-sources.csv")
columns = ["query", "selection", "channel", "video", "date", "seconds", "passage_start_seconds", "line_start_seconds", "passage",
           "source", "transcript", "transcript_type", "review"]
with output.open("w", encoding="utf-8", newline="") as stream:
    writer = csv.DictWriter(stream, fieldnames=columns)
    writer.writeheader()
    for item in passages:
        row = {"query": search_term, "selection": "live search" if live_search else "saved selection",
               **item, "review": ""}
        writer.writerow({k: spreadsheet_text(v) for k, v in row.items()})
print(f"Saved {len(passages)} rows to {output.name}.")

Saved 5 rows to arcmira-quote-sources.csv.
